# 00ter — Revue humaine des annotations de jardins

Examine les photos `PENDING` et leurs neuf caractéristiques produites par Gemini. Corrige les caractéristiques, puis décide **ACCEPTED**, **REJECTED** ou **UNCERTAIN**.

- Aucune mise à jour automatique : l'enregistrement nécessite un clic explicite par photo.
- **ACCEPTED** : `review_status='ACCEPTED'`, `reviewed=true` ; les neuf caractéristiques corrigées sont enregistrées.
- **REJECTED / UNCERTAIN** : `reviewed=false`, `dataset_split=NULL` ; ces photos ne sont pas utilisées pour l'entraînement.
- `dataset_split` reste `NULL` même après acceptation. La répartition TRAIN/VAL/TEST est une étape ultérieure.
- Aucun appel Gemini. Les anciennes photos déjà validées ne sont pas modifiées.
- Après validation, utiliser `00_Jardins_Etiquetage_FR_EN.ipynb` pour attribuer le type de jardin (`GARDEN_TYPE`).



In [ ]:
from pathlib import Path
import os, json
import psycopg
from dotenv import load_dotenv
from IPython.display import display, Image, Markdown, clear_output
import ipywidgets as widgets

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "ai-service/plant-ingestion/inputs/images").is_dir())
load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT / ".env.local", override=False)
DSN = os.environ["GARDNAIZE_ML_DATABASE_URL"].replace("postgresql+psycopg://", "postgresql://", 1)
IMAGES = (ROOT / "ai-service/plant-ingestion/inputs/images").resolve()
FEATURES = ("lawn", "terrace", "pathway", "trees", "shrubs", "fence", "low_wall", "stairs", "slope")
def connect(): return psycopg.connect(DSN)

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT g.image_id, g.image_uri, f.feature_values
            FROM horticulture.garden_image_candidate g
            JOIN horticulture.ml_garden_features f ON f.image_id = g.image_id
            WHERE g.review_status = 'PENDING' AND f.reviewed = false
            ORDER BY g.image_id
        """)
        pending = cur.fetchall()
print("Photos en attente de revue :", len(pending))



In [ ]:
def resolve_photo(uri):
    p = (ROOT / "ai-service/plant-ingestion" / uri).resolve()
    if not p.is_relative_to(IMAGES) or not p.is_file():
        return None
    return p

def review_one(image_id, uri, raw_features):
    features = raw_features if isinstance(raw_features, dict) else json.loads(raw_features)
    display(Markdown(f"### Photo #{image_id} — {Path(uri).name}"))
    photo = resolve_photo(uri)
    if photo is not None:
        display(Image(filename=str(photo), width=650))
    else:
        display(Markdown("**Photo introuvable ou chemin non autorisé : validation désactivée.**"))
    selectors = {}
    for key in FEATURES:
        value = features.get(key)
        if value not in (True, False, None):
            raise ValueError(f"Valeur invalide pour {key} : {value!r}")
        w = widgets.Dropdown(
            options=[("Oui / True", "true"), ("Non / False", "false"), ("Indéterminé / Null", "null")],
            value="null" if value is None else ("true" if value else "false"),
            description=key,
            style={"description_width": "120px"},
            layout=widgets.Layout(width="360px"))
        selectors[key] = w
    decision = widgets.Dropdown(
        options=[("— Choisir une décision —", ""),
                 ("ACCEPTED — photo et annotation vérifiées", "ACCEPTED"),
                 ("REJECTED — photo non exploitable", "REJECTED"),
                 ("UNCERTAIN — à revoir", "UNCERTAIN")],
        description="Décision", style={"description_width": "120px"},
        layout=widgets.Layout(width="550px"))
    button = widgets.Button(description=f"Enregistrer la revue #{image_id}", button_style="success")
    result = widgets.Output()
    def save(_):
        with result:
            clear_output(wait=True)
            if photo is None:
                print("Refus : photo non accessible.")
                return
            if not decision.value:
                print("Choisir explicitement ACCEPTED, REJECTED ou UNCERTAIN.")
                return
            corrected = {key: {"true": True, "false": False, "null": None}[w.value]
                         for key, w in selectors.items()}
            accepted = decision.value == "ACCEPTED"
            try:
                with connect() as conn:
                    with conn.cursor() as cur:
                        # Contrôle optimiste : ne pas écraser une revue faite ailleurs.
                        cur.execute("""
                            SELECT g.review_status, f.reviewed
                            FROM horticulture.garden_image_candidate g
                            JOIN horticulture.ml_garden_features f USING (image_id)
                            WHERE g.image_id = %s FOR UPDATE OF g, f
                        """, (image_id,))
                        state = cur.fetchone()
                        if state != ("PENDING", False):
                            print("Enregistrement annulé : état modifié depuis le chargement :", state)
                            return
                        cur.execute("""
                            UPDATE horticulture.ml_garden_features
                            SET feature_values = %s::jsonb, reviewed = %s
                            WHERE image_id = %s
                        """, (json.dumps(corrected), accepted, image_id))
                        cur.execute("""
                            UPDATE horticulture.garden_image_candidate
                            SET review_status = %s, dataset_split = NULL
                            WHERE image_id = %s
                        """, (decision.value, image_id))
                    conn.commit()
                button.disabled = True
                print(f"Photo #{image_id} enregistrée : {decision.value}, reviewed={accepted}, split=NULL")
            except Exception as exc:
                print("Échec de la revue :", type(exc).__name__, str(exc))
    button.on_click(save)
    display(widgets.VBox(list(selectors.values()) + [decision, button, result]))

for image_id, uri, features in pending:
    review_one(image_id, uri, features)



In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT g.review_status, f.reviewed, COUNT(*)
            FROM horticulture.garden_image_candidate g
            JOIN horticulture.ml_garden_features f USING (image_id)
            GROUP BY g.review_status, f.reviewed
            ORDER BY g.review_status, f.reviewed
        """)
        for status, reviewed, count in cur.fetchall():
            print(f"{status:10s} reviewed={reviewed}: {count}")
print("Étape suivante : 00_Jardins_Etiquetage_FR_EN.ipynb pour le label GARDEN_TYPE.")

